**Target Dataset**

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../../data/processed/project_04_target_returns.csv")
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Ticker", "Date"]).copy()

**Daily Return**

In [3]:
df["ret_1"] = df.groupby("Ticker")["Close"].pct_change()

**Momentum Features**

In [ ]:
df["ret_5"] = df.groupby("Ticker")["Close"].pct_change(5)
df["ret_10"] = df.groupby("Ticker")["Close"].pct_change(10)
df["ret_20"] = df.groupby("Ticker")["Close"].pct_change(20)


**Volatility Feature**

In [5]:
df["vol_5"] = (
    df.groupby("Ticker")["ret_1"]
    .rolling(5)
    .std()
    .reset_index(level=0, drop=True)
)

df["vol_20"] = (
    df.groupby("Ticker")["ret_1"]
    .rolling(20)
    .std()
    .reset_index(level=0, drop=True)
)

**Moving Average**

In [7]:
ma_10 = (
    df.groupby("Ticker")["Close"]
    .rolling(10)
    .mean()
    .reset_index(level=0, drop=True)
)

ma_20 = (
    df.groupby("Ticker")["Close"]
    .rolling(20)
    .mean()
    .reset_index(level=0, drop=True)
)

ma_50 = (
    df.groupby("Ticker")["Close"]
    .rolling(50)
    .mean()
    .reset_index(level=0, drop=True)
)

df["ma_10_ratio"] = df["Close"] / ma_10 - 1
df["ma_20_ratio"] = df["Close"] / ma_20 - 1
df["ma_50_ratio"] = df["Close"] / ma_50 - 1

**Volume Features**

In [8]:
df["dollar_vol"] = df["Close"] * df["Volume"]

vol_mean_20 = (
    df.groupby("Ticker")["Volume"]
    .rolling(20)
    .mean()
    .reset_index(level=0, drop=True)
)

vol_std_20 = (
    df.groupby("Ticker")["Volume"]
    .rolling(20)
    .std()
    .reset_index(level=0, drop=True)
)

df["vol_z_20"] = (df["Volume"] - vol_mean_20) / vol_std_20

In [9]:
feature_cols = [
    "Date", "Ticker",
    "ret_1", "ret_5", "ret_10", "ret_20",
    "vol_5", "vol_20",
    "ma_10_ratio", "ma_20_ratio", "ma_50_ratio",
    "dollar_vol", "vol_z_20",
    "fwd_ret_5"
]

df_features = df[feature_cols].copy()

**Drop NaNs**

In [10]:
df_features = df_features.dropna().copy()

In [12]:
df_features.shape

(175521, 14)

In [14]:
df_features.isna().sum()

Date           0
Ticker         0
ret_1          0
ret_5          0
ret_10         0
ret_20         0
vol_5          0
vol_20         0
ma_10_ratio    0
ma_20_ratio    0
ma_50_ratio    0
dollar_vol     0
vol_z_20       0
fwd_ret_5      0
dtype: int64

In [15]:
df_features.describe()

,Date,ret_1,ret_5,ret_10,ret_20,vol_5,vol_20,ma_10_ratio,ma_20_ratio,ma_50_ratio,dollar_vol,vol_z_20,fwd_ret_5
count,175521,175521.000000,175521.000000,175521.000000,175521.000000,175521.000000,175521.000000,175521.000000,175521.000000,175521.000000,1.755210e+05,175521.000000,175521.000000
mean,2007-07-26 12:11:45.638641,0.000852,0.004202,0.008324,0.016658,0.017280,0.018547,0.003007,0.006412,0.016753,1.545435e+09,0.001540,0.004194
min,1985-03-13 00:00:00,-0.518475,-0.587736,-0.635157,-0.670877,0.000000,0.000000,-0.525124,-0.580128,-0.640972,0.000000e+00,-3.349143,-0.587736
25%,1998-03-20 00:00:00,-0.008801,-0.018541,-0.024428,-0.030912,0.008934,0.011127,-0.013554,-0.018158,-0.024224,1.101031e+08,-0.695818,-0.018555
50%,2008-05-30 00:00:00,0.000106,0.003613,0.007118,0.014124,0.013687,0.015339,0.003358,0.006900,0.016975,4.984479e+08,-0.232317,0.003604
75%,2017-06-13 00:00:00,0.010216,0.025846,0.039103,0.060022,0.021035,0.021873,0.019823,0.031392,0.057376,1.273239e+09,0.476343,0.025837
max,2026-03-06 00:00:00,0.424952,1.555256,1.459592,2.150407,0.244869,0.154420,0.785541,1.053111,1.737570,1.543779e+11,4.246475,1.555256
std,NaN,0.022068,0.048114,0.066823,0.094892,0.014000,0.012125,0.035655,0.051621,0.083455,3.975132e+09,1.040853,0.048087


In [16]:
df_features.head()

,Date,Ticker,ret_1,ret_5,ret_10,ret_20,vol_5,vol_20,ma_10_ratio,ma_20_ratio,ma_50_ratio,dollar_vol,vol_z_20,fwd_ret_5
49,1985-03-13,AAPL,-0.055790,-0.117225,-0.134128,-0.269486,0.058919,0.037152,-0.078848,-0.143323,-0.217203,2.430735e+07,-0.225229,0.022112
50,1985-03-14,AAPL,0.000000,-0.018036,-0.120045,-0.234401,0.039775,0.036553,-0.067125,-0.131939,-0.213707,2.338588e+07,-0.208773,0.040356
51,1985-03-15,AAPL,0.040356,0.051920,-0.090526,-0.180383,0.040566,0.038192,-0.020011,-0.087843,-0.178542,1.826865e+07,-0.547399,-0.017536
52,1985-03-18,AAPL,0.010802,0.028843,-0.095324,-0.183270,0.038625,0.038084,0.001022,-0.068352,-0.166318,1.269978e+07,-0.912524,-0.055956
53,1985-03-19,AAPL,-0.038720,-0.045521,-0.151865,-0.203609,0.038669,0.038639,-0.020867,-0.094054,-0.194922,1.677565e+07,-0.602361,0.025457


In [27]:
df_features.to_csv(
    "../../../data/processed/project_04_features.csv",
    index=False
)

**Feature → target correlation**

In [18]:
feature_cols = [
"ret_1","ret_5","ret_10","ret_20",
"vol_5","vol_20",
"ma_10_ratio","ma_20_ratio","ma_50_ratio",
"dollar_vol","vol_z_20"
]

df_features[feature_cols + ["fwd_ret_5"]].corr()["fwd_ret_5"].sort_values()

ma_10_ratio   -0.048603
ret_5         -0.044719
ma_20_ratio   -0.038048
ret_10        -0.032754
ret_1         -0.029688
ma_50_ratio   -0.026003
ret_20        -0.017299
vol_z_20       0.006474
dollar_vol     0.006480
vol_5          0.027887
vol_20         0.044420
fwd_ret_5      1.000000
Name: fwd_ret_5, dtype: float64

**Longer Horizon Returns and Correlation**

In [28]:
df["ret_60"] =df.groupby("Ticker")["Close"].pct_change(60)
df["ret_120"] = df.groupby("Ticker")["Close"].pct_change(120)

**Feature_cols_v2**

In [29]:
feature_cols_v2 = [
    "Date", "Ticker",
    "ret_1", "ret_5", "ret_10", "ret_20",
    "ret_60", "ret_120",   # NEW
    "vol_5", "vol_20",
    "ma_10_ratio", "ma_20_ratio", "ma_50_ratio",
    "dollar_vol", "vol_z_20",
    "fwd_ret_5"
]

In [30]:
df_features_v2 = df[feature_cols_v2].copy()
df_features_v2 = df_features_v2.dropna().copy()

In [31]:
df_features_v2.to_csv(
    "../../../data/processed/project_04_features_v2.csv",
    index=False
)

**Correlation**

In [33]:
feature_only_cols = [
    col for col in feature_cols_v2 
    if col not in ["Date", "Ticker", "fwd_ret_5"]
]

corr = df_features_v2[feature_only_cols + ["fwd_ret_5"]].corr()["fwd_ret_5"]

corr.sort_values()

ma_10_ratio   -0.047844
ret_5         -0.043635
ma_20_ratio   -0.038567
ret_10        -0.033930
ret_1         -0.028176
ma_50_ratio   -0.027540
ret_20        -0.017983
ret_60        -0.014775
ret_120        0.004677
vol_z_20       0.005811
dollar_vol     0.007685
vol_5          0.027048
vol_20         0.043724
fwd_ret_5      1.000000
Name: fwd_ret_5, dtype: float64

The inclusion of longer-horizon features (ret_60 and ret_120) reveals that mean reversion persists up to 60 days, while weak momentum emerges at 120 days; however, these signals are significantly weaker than short-term features and may have limited impact on model performance.